In [ ]:
%load_ext autoreload
%autoreload 2

## Data Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.preprocessing import OneHotEncoder

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_validate, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay
from collections import Counter

In [ ]:
df = pd.read_excel("../data/Training-data.xlsx")
print(df.head())

In [ ]:
test1 = pd.read_excel("../data/Test-data-1.xlsx")

In [ ]:
test2 = pd.read_excel("../data/Test-data-2.xlsx")

## Data Preparation (non-learned steps)

### Irrelevant Features

In [ ]:
df1 = df.drop(columns=["tcprtt"])

### Missing Values

In [ ]:
# No empty values for now

### Duplicated Values

In [ ]:
df2 = df1.drop(columns=["is_ftp_login"])

### Feature Attribute Selection

In [ ]:
df3 = df2.drop(columns=['dwin'])

### Feature Engineering

In [ ]:
# Empty for now

### Class Imbalance

In [ ]:
# Empty for now

## Cross Validation setup

In [ ]:
from sklearn.model_selection import StratifiedKFold, KFold, ShuffleSplit

skf = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=67
)

y = df3['label']
X = df3.drop(columns=['label'])


numeric_cols = X.select_dtypes(include=['number']).columns
categorical_cols = X.select_dtypes(include=['string']).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(sparse_output=False, handle_unknown="ignore"), categorical_cols)
    ]
)

def evaluate_cv(model, X, y, cv):
    y_pred = cross_val_predict(
        model,
        X,
        y,
        cv=cv,
        n_jobs=-1
    )

    print(classification_report(y, y_pred))

    ConfusionMatrixDisplay.from_predictions(
        y, y_pred,
        labels=[0, 1]
    )

    plt.show()

## k-NN Classifier

### k-NN: Build a pipeline 

In [ ]:
# k_values = [1,3,7,11,17,21]
k_values = [7]

def make_knn_pipeline(k):
    return Pipeline([
        ("preprocessor", preprocessor),                 
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

### k-NN: Evaluate different k-values with CV

In [ ]:
cv_records = []
for k in k_values:
    print("Training on k = {}".format(k))
    pipe = make_knn_pipeline(k)
    scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy", n_jobs=-1)
    cv_records.append({"k": k, "mean_cv_acc": scores.mean(), "std_cv_acc": scores.std()})

### k-NN: Select the champion model

In [ ]:
cv_df = pd.DataFrame(cv_records).sort_values("mean_cv_acc", ascending=False).reset_index(drop=True)
print("\nCross-validation accuracy by k (training set):")
print(cv_df.to_string(index=False))

best_k = cv_df.loc[0, 'k']
print("Best k: {}".format(best_k))


### k-NN: Generating the performance report on the full dataset

In [ ]:
pipe = make_knn_pipeline(best_k)
evaluate_cv(pipe, X, y, skf)

## Decision Tree classifier

### DT: Build a pipeline

In [ ]:
def make_dt_pipeline():
    return Pipeline([
        ("preprocessor", preprocessor),                 
        ("dt", DecisionTreeClassifier())
    ])

### DT: Evaluate different hyperparameters

In [ ]:
parameters = {
    'dt__criterion': ('gini', 'entropy'),
    'dt__min_samples_split' : list(range(1,11))
    }

gscv = GridSearchCV(
    estimator=make_dt_pipeline(),
    param_grid=parameters,
    cv=skf,
    scoring='accuracy',
    n_jobs = -1,
    verbose = 2
)

best_dtc = gscv.fit(X, y)

### DT: Select the champion model

In [ ]:
print(best_dtc.best_params_)
print(best_dtc.best_score_)
final_model = best_dtc.best_estimator_

evaluate_cv(final_model, X, y, skf)

## Naive-Bayes classifier

### NB: Build a pipeline

In [ ]:
def make_nb_pipeline():
    return Pipeline([
        ("preprocessor", preprocessor),                 
        ("nb", GaussianNB())
    ])

### NB: Evaluate different hyperparameters

In [ ]:
parameters = {
    "nb__var_smoothing": [
        # 1e-12, 1e-11, 1e-10, 1e-9,
        # 1e-8, 1e-7, 1e-6
    
        1e-6,
        1e-5,
        1e-4,
        1e-3,
        1e-2,
        1e-1,
        1
    ]
}

gscv = GridSearchCV(
    estimator=make_nb_pipeline(),
    param_grid=parameters,
    cv=skf,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)
best_nb = gscv.fit(X, y)

### NB: Select the champion model

In [ ]:
print(best_nb.best_params_)
print(best_nb.best_score_)
final_model = best_nb.best_estimator_

evaluate_cv(final_model, X, y, skf)